# 🚀 Fine-Tuning Transformers for Customer Complaint Classification

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Khitishnayak/Customer-Complaint-Classification/blob/main/notebooks/customer_complaint_finetuning.ipynb)

This notebook fine-tunes a **Transformer (DistilBERT)** to classify customer complaints into multi-class business departments with deep contextual understanding.

### Key Objectives:
1. Load cross-industry customer complaints (Banking, Tech, E-Commerce, Travel, Telecom).
2. Tokenize text narratives with `distilbert-base-uncased`.
3. Fine-tune using Hugging Face `Trainer` with GPU acceleration (`fp16`).
4. Evaluate with Precision, Recall, Macro F1, and Confusion Matrix.
5. Save the fine-tuned model weights for deployment in the Streamlit app.

## 1. Environment & GPU Verification

In [ ]:
# Install dependencies if running on Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running on Google Colab! Installing Hugging Face libraries...")
    !pip install -q transformers datasets evaluate accelerate scikit-learn seaborn matplotlib
except ImportError:
    IN_COLAB = False
    print("Running locally on Jupyter!")

import torch
print(f"PyTorch Version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Notice: No CUDA GPU detected. Training on CPU will be slower. If on Colab, go to Runtime > Change runtime type > T4 GPU.")

## 2. Load and Prepare the Dataset

In [ ]:
import os
import pandas as pd
import numpy as np

# If in Colab, clone the repo to fetch preprocessed data
if IN_COLAB:
    if not os.path.exists("Customer-Complaint-Classification"):
        !git clone https://github.com/Khitishnayak/Customer-Complaint-Classification.git
    data_path = "Customer-Complaint-Classification/data/processed/complaints_multidomain_master.csv"
else:
    data_path = "../data/processed/complaints_multidomain_master.csv"

if not os.path.exists(data_path):
    # Fallback to local sample or scaled dataset
    data_path = "../data/processed/complaints_sample.csv" if os.path.exists("../data/processed/complaints_sample.csv") else "data/processed/complaints_sample.csv"

df = pd.read_csv(data_path)
print(f"Dataset loaded: {len(df):,} total rows across {df[category].nunique()} categories.")
print("\nCategory breakdown:")
print(df["category"].value_counts())

## 3. Label Encoding & Train/Validation/Test Splits

In [ ]:
from sklearn.model_selection import train_test_split

# Clean text and drop NaN
df = df.dropna(subset=["text", "category"]).copy()
df["text"] = df["text"].astype(str)

# Create label mappings
categories = sorted(df["category"].unique())
label2id = {cat: i for i, cat in enumerate(categories)}
id2label = {i: cat for i, cat in enumerate(categories)}
df["label"] = df["category"].map(label2id)

# For faster training in demo/Colab, you can sample e.g. 15,000 to 30,000 rows if needed
sample_size = min(len(df), 25000)  # Change to len(df) for full scale training
df_sampled = df.groupby("label", group_keys=False).apply(lambda x: x.sample(min(len(x), sample_size // len(categories)), random_state=42))

# Stratified Split: 80% Train, 10% Validation, 10% Test
train_df, temp_df = train_test_split(df_sampled, test_size=0.20, random_state=42, stratify=df_sampled["label"])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df["label"])

print(f"Train samples: {len(train_df):,}")
print(f"Validation samples: {len(val_df):,}")
print(f"Test samples: {len(test_df):,}")

## 4. Tokenization with Hugging Face
We use `distilbert-base-uncased`, which provides 97% of BERT-base performance with 40% fewer parameters and 60% faster inference.

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
        padding="max_length"
    )

# Convert DataFrames to Hugging Face Datasets
train_dataset = Dataset.from_pandas(train_df[["text", "label"]]).map(tokenize_fn, batched=True)
val_dataset = Dataset.from_pandas(val_df[["text", "label"]]).map(tokenize_fn, batched=True)
test_dataset = Dataset.from_pandas(test_df[["text", "label"]]).map(tokenize_fn, batched=True)

print("Tokenization complete!")

## 5. Model Setup & Evaluation Metrics

In [ ]:
from transformers import AutoModelForSequenceClassification
import evaluate

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(categories),
    id2label=id2label,
    label2id=label2id
)

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    macro_f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "macro_f1": macro_f1}

## 6. Training Arguments & Fine-Tuning Loop

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./distilbert-complaints-checkpoints",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# Start Fine-Tuning!
trainer.train()

## 7. Unseen Test Set Evaluation & Confusion Matrix

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Predict on test set
test_preds_raw = trainer.predict(test_dataset)
test_preds = np.argmax(test_preds_raw.predictions, axis=-1)
true_labels = test_df["label"].values

print("=== Fine-Tuned Transformer Classification Report ===\n")
print(classification_report(true_labels, test_preds, target_names=categories))

# Plot Confusion Matrix
cm = confusion_matrix(true_labels, test_preds)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=categories, yticklabels=categories)
plt.xlabel("Predicted Category")
plt.ylabel("Actual Category")
plt.title("Confusion Matrix - Fine-Tuned DistilBERT")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 8. Real-Time Inference Pipeline

In [ ]:
import torch.nn.functional as F

def predict_complaint(text, top_k=3):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1).squeeze().cpu().numpy()
    
    top_indices = np.argsort(probs)[::-1][:top_k]
    print(f"\nComplaint: {text}")
    for idx in top_indices:
        print(f"  → {id2label[idx]}: {probs[idx]*100:.2f}%")

# Test sample complaints across domains
predict_complaint("My flight from Chicago was delayed for 8 hours and my luggage never showed up.")
predict_complaint("I was charged an unexpected overdraft fee on my checking account despite having funds.")
predict_complaint("My Amazon prime package says delivered but it was never left at my door.")
predict_complaint("The collection agency keeps harassing me at 6 AM over a medical bill that was already cleared.")

## 9. Export Model Weights for Streamlit App

In [ ]:
SAVE_DIR = "./fine_tuned_distilbert_complaints"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Fine-tuned model successfully saved to: {SAVE_DIR}")

if IN_COLAB:
    # Zip the model directory so you can download it easily
    !zip -r fine_tuned_distilbert.zip ./fine_tuned_distilbert_complaints
    print("\nGenerated fine_tuned_distilbert.zip! Download it from the Colab file browser.")